In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- hybrid_score_concat ---
FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_BEFORE = [
    SimpleNamespace(to_frame=lambda: pd.DataFrame({"document_id":[0],"score":[0.9]})),
    SimpleNamespace(to_frame=lambda: pd.DataFrame({"document_id":[1],"score":[0.7]})),
]
FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_GEN = [
    SimpleNamespace(to_frame=lambda: pl.DataFrame({"document_id":[0],"score":[0.9]})),
    SimpleNamespace(to_frame=lambda: pl.DataFrame({"document_id":[1],"score":[0.7]})),
]

# --- hybrid_score_frame ---
FIX_HYBRID_SCORE_FRAME_ARGS = dict(
    language_model_name="bert",
    citation_to_language_candidates=0.123456,
    citation_to_language=0.654321,
    language_to_citation_candidates=0.111111,
    language_to_citation=0.999999,
)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_hybrid_score_concat(hybrid_scores):
    return pd.concat([hybrid_score.to_frame() for hybrid_score in hybrid_scores], ignore_index=True)
    return None

def before_hybrid_score_frame(language_model_name, citation_to_language_candidates, citation_to_language, language_to_citation_candidates, language_to_citation):
    return pd.DataFrame(
        {
            "Language Model": language_model_name,
            "Citation -> Language Candidates": round(
                citation_to_language_candidates, ndigits=3
            ),
            "Citation -> Language Final": round(citation_to_language, ndigits=3),
            "Language -> Citation Candidates": round(
                language_to_citation_candidates, ndigits=3
            ),
            "Language -> Citation Final": round(language_to_citation, ndigits=3),
        },
        index=[0],
    )

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_hybrid_score_concat(hybrid_scores):
    return pl.concat([hybrid_score.to_frame() for hybrid_score in hybrid_scores], how="vertical")
    return None

def gen_hybrid_score_frame(language_model_name, citation_to_language_candidates, citation_to_language, language_to_citation_candidates, language_to_citation):
    return pl.DataFrame(
        {
            "Language Model": [language_model_name],
            "Citation -> Language Candidates": [round(citation_to_language_candidates, ndigits=3)],
            "Citation -> Language Final": [round(citation_to_language, ndigits=3)],
            "Language -> Citation Candidates": [round(language_to_citation_candidates, ndigits=3)],
            "Language -> Citation Final": [round(language_to_citation, ndigits=3)],
        }
    )

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: hybrid_score_frame ===

# L1 smoke – generated
try:
    _r = gen_hybrid_score_frame(**FIX_HYBRID_SCORE_FRAME_ARGS)
    print("✅ L1 smoke gen_hybrid_score_frame: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_hybrid_score_frame: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_hybrid_score_frame(**FIX_HYBRID_SCORE_FRAME_ARGS)
    print("✅ L1 smoke before_hybrid_score_frame: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_hybrid_score_frame: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_hybrid_score_frame(**FIX_HYBRID_SCORE_FRAME_ARGS)
    _rg = gen_hybrid_score_frame(**FIX_HYBRID_SCORE_FRAME_ARGS)
    compare(_rb, _rg, "hybrid_score_frame")
except Exception as _e:
    print(f"❌ L2 equivalence hybrid_score_frame: setup error — {type(_e).__name__}: {_e}")

# L3 — negative / rounding edge values
try:
    _edge_args = dict(FIX_HYBRID_SCORE_FRAME_ARGS)
    _edge_args.update(citation_to_language_candidates=0.0, language_to_citation=1.0)
    _rb = before_hybrid_score_frame(**_edge_args)
    _rg = gen_hybrid_score_frame(**_edge_args)
    compare(_rb, _rg, "L3 hybrid_score_frame edge values")
except Exception as _e:
    print(f"❌ L3 hybrid_score_frame edge values: {type(_e).__name__}: {_e}")
